In [ ]:
"""
TALLER DE PROCESAMIENTO DE DATOS - MACHINE LEARNING
Universidad Libre - Base de datos VENTAS_NL

Este script genera EN VIVO todas las tablas y graficas que aparecen en el
PDF de entrega: skewness, distribuciones, matriz de correlacion,
comparacion de modelos e importancia de variables.

Integrantes:
- Jarol Steven Gutierrez Gordillo (LIDER)
- Daniel Mauricio Agreda Aguilar
Fecha: 15 de agosto de 2026
"""

In [ ]:
# ============ 0. INSTALAR/IMPORTAR LIBRERIAS ============
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 110

In [ ]:
RUTA_ARCHIVO = "/content/VENTAS_NL.xlsx"  # Ruta en Colab tras subir el archivo

In [ ]:
# ============ 1. CARGA Y LIMPIEZA ============
df = pd.read_excel(RUTA_ARCHIVO)
n_original = len(df)

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df['id'] = df['id'].astype(int)
df['EDAD'] = df['EDAD'].astype(int)
df['GENERO'] = df['GENERO'].astype(str).str.strip().str.upper()
df['SIZE'] = df['SIZE'].astype(int)
for col in ['YEARINCOME', 'ventas', 'costo venta', 'DESCUENTOS']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
print("="*70)
print("RESUMEN DE LIMPIEZA")
print("="*70)
print(f"Registros originales : {n_original}")
print(f"Duplicados eliminados: {n_dup}")
print(f"Registros finales    : {len(df)}")

In [ ]:
# ============ 2. DATOS FALTANTES + SKEWNESS + IMPUTACION ============
cols_faltantes = ['YEARINCOME', 'ventas', 'costo venta', 'DESCUENTOS']
reporte = []

In [ ]:
# Guardamos copia ANTES de imputar, para graficar las distribuciones originales
df_antes = df.copy()

In [ ]:
for col in cols_faltantes:
    n_missing = df[col].isna().sum()
    sk = skew(df[col].dropna())
    if abs(sk) <= 0.5:
        metodo, valor = 'Media', df[col].mean()
    else:
        metodo, valor = 'Mediana', df[col].median()
    df[col] = df[col].fillna(valor)
    reporte.append({'Columna': col, 'Faltantes': int(n_missing),
                     '% Faltantes': round(n_missing/len(df)*100, 2),
                     'Skewness': round(sk, 3), 'Metodo': metodo,
                     'Valor usado': round(valor, 4)})

In [ ]:
reporte_df = pd.DataFrame(reporte)
print("\n" + "="*70)
print("TABLA 1: IMPUTACION DE DATOS FALTANTES")
print("="*70)
display(reporte_df)  # en Colab 'display' funciona directo

In [ ]:
# --- GRAFICA 1: Skewness por variable ---
fig, ax = plt.subplots(figsize=(7, 4))
colors_bar = ['#d62728' if abs(v) > 0.5 else '#2ca02c' for v in reporte_df['Skewness']]
bars = ax.bar(reporte_df['Columna'], reporte_df['Skewness'], color=colors_bar)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.axhline(-0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title('Coeficiente de Asimetria (Skewness) por Variable', fontsize=12, fontweight='bold')
ax.set_ylabel('Skewness')
for bar, v in zip(bars, reporte_df['Skewness']):
    ax.text(bar.get_x()+bar.get_width()/2, v + (0.05 if v >= 0 else -0.15),
            f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- GRAFICA 2: Distribuciones antes de imputar ---
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
for ax, col in zip(axes.flat, cols_faltantes):
    ax.hist(df_antes[col].dropna(), bins=40, color='#1f4e78', alpha=0.85)
    ax.set_title(col, fontsize=10)
plt.suptitle('Distribucion de variables con datos faltantes (antes de imputar)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============ 3. ESTANDARIZACION ============
scaler_std = StandardScaler()
df[['YEARINCOME_std', 'ventas_std']] = scaler_std.fit_transform(df[['YEARINCOME', 'ventas']])

In [ ]:
scaler_mm = MinMaxScaler()
df['costo_venta_norm'] = scaler_mm.fit_transform(df[['costo venta']])

In [ ]:
print("\n" + "="*70)
print("TABLA 2: EJEMPLO DE VARIABLES ESTANDARIZADAS")
print("="*70)
display(df[['YEARINCOME', 'YEARINCOME_std', 'ventas', 'ventas_std',
            'costo venta', 'costo_venta_norm']].head(10))

In [ ]:
# ============ 4. INGENIERIA DE CARACTERISTICAS ============
# Variable 1: ventas netas despues de descuento
df['ventas_netas'] = df['ventas'] * (1 - df['DESCUENTOS'])

In [ ]:
# Variable 2 (no obvia): indice de gasto relativo al ingreso anual
df['indice_gasto_relativo'] = df['ventas'] / df['YEARINCOME']

In [ ]:
print("\n" + "="*70)
print("TABLA 3: NUEVAS VARIABLES CREADAS")
print("="*70)
display(df[['ventas', 'DESCUENTOS', 'ventas_netas', 'YEARINCOME',
            'indice_gasto_relativo']].head(10))

In [ ]:
# ============ 5. MATRIZ DE CORRELACION ============
cols_corr = ['EDAD', 'SIZE', 'YEARINCOME', 'ventas', 'costo venta',
             'DESCUENTOS', 'ventas_netas', 'indice_gasto_relativo']
corr_matrix = df[cols_corr].corr().round(3)

In [ ]:
print("\n" + "="*70)
print("TABLA 4: MATRIZ DE CORRELACION")
print("="*70)
display(corr_matrix)

In [ ]:
# --- GRAFICA 3: Heatmap de correlacion ---
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, cbar_kws={'label': 'Correlacion'})
ax.set_title('Matriz de Correlacion', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============ 6. MODELOS: RANDOM FOREST vs GRADIENT BOOSTING ============
model_df = df.copy()
model_df['GENERO_enc'] = model_df['GENERO'].map({'M': 0, 'H': 1})

In [ ]:
X = model_df[['EDAD', 'GENERO_enc', 'SIZE', 'YEARINCOME', 'DESCUENTOS']]
y = model_df['ventas']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
modelos = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42)
}

In [ ]:
resultados = []
importancias = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'R2': round(r2_score(y_test, pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2),
        'MAE': round(mean_absolute_error(y_test, pred), 2)
    })
    importancias[nombre] = dict(zip(X.columns, modelo.feature_importances_.round(4)))

In [ ]:
resultados_df = pd.DataFrame(resultados)
fi_df = pd.DataFrame(importancias).reset_index().rename(columns={'index': 'Variable'})

In [ ]:
print("\n" + "="*70)
print("TABLA 5: RESULTADOS DE LOS MODELOS")
print("="*70)
display(resultados_df)

In [ ]:
print("\n" + "="*70)
print("TABLA 6: IMPORTANCIA DE VARIABLES")
print("="*70)
display(fi_df)

In [ ]:
# --- GRAFICA 4: Comparacion de modelos (R2 y RMSE) ---
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(resultados_df['Modelo'], resultados_df['R2'], color=['#1f4e78', '#2ca02c'])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('R2 por modelo')
for i, v in enumerate(resultados_df['R2']):
    axes[0].text(i, v + (0.001 if v >= 0 else -0.003), f'{v:.4f}', ha='center', fontsize=9)

In [ ]:
axes[1].bar(resultados_df['Modelo'], resultados_df['RMSE'], color=['#1f4e78', '#2ca02c'])
axes[1].set_title('RMSE por modelo')
for i, v in enumerate(resultados_df['RMSE']):
    axes[1].text(i, v, f'{v:,.0f}', ha='center', fontsize=9)

In [ ]:
plt.suptitle('Comparacion de Modelos: Random Forest vs Gradient Boosting', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- GRAFICA 5: Importancia de variables ---
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(fi_df))
width = 0.35
ax.bar(x - width/2, fi_df['Random Forest'], width, label='Random Forest', color='#1f4e78')
ax.bar(x + width/2, fi_df['Gradient Boosting'], width, label='Gradient Boosting', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(fi_df['Variable'])
ax.set_title('Importancia de Variables', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("PROCESO COMPLETADO - Todas las tablas y graficas fueron generadas arriba")
print("="*70)